In [1]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI bot. Your name is {name}."),
        ("human", "Hello, how are you doing?"),
        ("ai", "I'm doing well, thanks!"),
        ("human", "{user_input}"),
    ]
)

prompt_value = template.invoke(
    {
        "name": "Bob",
        "user_input": "What is your name?",
    }
)



/home/jazil/miniconda3/envs/bpjs-rag-bot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
prompt_txt = template.format(name="jazi", user_input="Whats your name?")
print(type(prompt_txt))
print(prompt_txt)

<class 'str'>
System: You are a helpful AI bot. Your name is jazi.
Human: Hello, how are you doing?
AI: I'm doing well, thanks!
Human: Whats your name?


In [14]:
from langchain_core.prompts import MessagesPlaceholder

prompt = MessagesPlaceholder("history")
# prompt.format_messages()  # raises KeyError

prompt = MessagesPlaceholder("history", optional=True)
prompt.format_messages()  # returns empty list []

prompt.format_messages(
    history=[
        ("system", "You are an AI assistant."),
        ("human", "Hello!"),
    ]
)
# -> [
#     SystemMessage(content="You are an AI assistant."),
#     HumanMessage(content="Hello!"),
# ]

# print(prompt.format_messages)

[SystemMessage(content='You are an AI assistant.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello!', additional_kwargs={}, response_metadata={})]

In [18]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant."),
        MessagesPlaceholder("history"),
        ("human", "{question}"),
    ]
)
prompt.invoke(
    {
        "history": [("human", "what's 5 + 2"), ("ai", "5 + 2 is 7")],
        "question": "now multiply that by 4",
    }
)
# -> ChatPromptValue(messages=[
#     SystemMessage(content="You are a helpful assistant."),
#     HumanMessage(content="what's 5 + 2"),
#     AIMessage(content="5 + 2 is 7"),
#     HumanMessage(content="now multiply that by 4"),
# ])

print(prompt.format(history=[("human", "what's 5 + 2"), ("ai", "5 + 2 is 7")], question="now multiply that by 4"))

System: You are a helpful assistant.
Human: what's 5 + 2
AI: 5 + 2 is 7
Human: now multiply that by 4


In [25]:
# CHAIN

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain.chains.base import Chain

from src.core.config import settings

llm = ChatGoogleGenerativeAI(
    model=settings.GENAI_MODEL,
    api_key=settings.GOOGLE_API_KEY,
    temperature=0.3
)

prompt = ChatPromptTemplate([
    ("sistem","Kamu adalah asisten AI"),
    ("human","{input}")
])

chain = prompt | llm | StrOutputParser()



ModuleNotFoundError: No module named 'langchain.chains'

In [2]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

from src.core.config import settings
import google.generativeai as genai
from google import genai

client = genai.Client()

print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

print("List of models that support embedContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "embedContent":
            print(m.name)


List of models that support generateContent:

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
List of models that support embedContent:

mode

### **UNDERSTANDING ABOUT CHAIN, RUNNABLE, INVOKE, PROMPT TEMPLATE, OUTPUT PARSER**
#### RUNNABLE
Runnable adalah istilah yang digunakan di dalam langchain terhadap setiap hal yang bisa dijalankan. Misalkan prompt, llm, output parser dan sebagainya.

#### CHAIN
Masalahnya, setiap runnable punya tipe input dan output berbeda sehingga sulit untuk mengintegrasikanya dalam satu proses yang terhubung. Untuk itu, **chain** digunakan dalam hal ini. Chain memungkinkan untuk menjalankan secara terhubung setiap runnable sehingga yang secara otomatis membuatnya kompatibel satu sama lain. 

#### INVOKE
Invoke adalah semacam tombol start yang digunakan untuk menjalankan runnable atau chain

In [8]:
# Example without langchain at all
from google import genai
from src.core.config import settings

# Prompt
instruction = "Berikan puisi tentang topik hujan"

client = genai.Client()

response = client.models.generate_content(
    model=settings.GENAI_MODEL,
    contents=instruction
)
print(response.text)


Tentu, ini dia sebuah puisi tentang hujan:

**Hujan**

Langit kelabu, tirai mulai terbentang,
Setitik air jatuh, melodi pertama datang.
Rintik perlahan, menyentuh dedaunan,
Menyapa bumi yang lama merindukan.

Kini deras, membasuh jalanan basah,
Mengalir di selokan, air pun berpisah.
Daun-daun berkilau, memancarkan pesona,
Dunia terguyur, seakan baru bermula.

Aroma tanah basah, menguar menenangkan,
Suara gemericik atap, irama mengalun.
Di balik jendela, kutatap dunia yang basah,
Sebuah jeda, merenung, tanpa resah.

Bukan hanya air, yang jatuh dari langit,
Namun kehidupan, tunas yang bangkit.
Mencuci debu kota, juga resah di hati,
Hujan, pembawa berkah, membersihkan sejati.


In [16]:
# Example with PromptTemplate, Chain, and Invoke

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from src.core.config import settings

# Prompt
instruction = "Berikan puisi tentang topik {topic}"

# llm
llm = ChatGoogleGenerativeAI(
    model = settings.GENAI_MODEL,
    api_key = settings.GOOGLE_API_KEY,
    temperature = 0.3
)

# prompt
prompt = ChatPromptTemplate([
    ("human", instruction)
])

chain = prompt | llm | StrOutputParser()

response = chain.invoke({"topic":"hujan"})
print(response)
prompt_invoke = prompt.invoke({"topic":"hujan"})
print("\n")
print(prompt_invoke)



Ketika langit kelabu mulai menyapa,
Awan berarak, janji basah tiba.
Rintik pertama, lembut menyapa,
Menari di atap, melodi syahdu tercipta.

Tetes demi tetes, membasahi bumi,
Mencuci dedaunan, membersihkan debu.
Mengalir di jalan, membentuk sungai mini,
Membawa aroma tanah yang baru.

Ia adalah berkah, anugerah dari atas,
Menyirami ladang, menghidupkan tunas.
Memberi kehidupan pada yang haus,
Menghijaukan dunia, tanpa batas.

Di balik tirainya, ada kedamaian,
Mengajak jiwa untuk merenung.
Duduk di jendela, dengan secangkir teh hangat,
Mendengar bisikan alam yang tak terhitung.

Oh, hujan, kau pembawa kehidupan,
Penyegar jiwa, penenang pikiran.
Dalam setiap tetesmu, ada keindahan,
Sebuah simfoni alam, abadi dalam ingatan.


messages=[HumanMessage(content='Berikan puisi tentang topik hujan', additional_kwargs={}, response_metadata={})]


In [17]:
response_without_chain = llm.invoke("Berikan puisi tentang hujan")
print(response_without_chain.content)

Tentu, ini puisi tentang hujan:

**Simfoni Hujan**

Langit kelabu, awan berarak pelan,
Membawa pesan dari kejauhan.
Rintik pertama, menari di daun,
Memecah sunyi, irama yang merdu.

Lalu derasnya, membasahi atap,
Mengetuk jendela, tanpa henti, tak henti.
Derai air jatuh, di jalanan beraspal,
Mencipta genangan, cermin dunia yang basah.

Aroma tanah basah, bangkit menyapa,
Segarkan udara, setelah lama dahaga.
Pohon-pohon menari, daunnya berkilau,
Debu terbasuh, dunia kembali syahdu.

Ada kisah tersembunyi, dalam setiap tetes,
Melodi syahdu, pengantar lamunan.
Kadang ia tangis, langit yang berduka,
Kadang ia berkah, pembawa harapan.

Hujan, kau adalah kehidupan yang jatuh,
Pembasuh luka, penumbuh tunas baru.
Walau dinginmu menusuk, namun kau perlu,
Simfoni alam, abadi selalu.
